In [31]:
from structure_tensor.h5_io import load_h5_datasets

from pathlib import Path
import matplotlib.pyplot as plt

import numpy as np
from scipy import ndimage as ndi

In [ ]:
file_path_vec = Path("../data/analysed/285_uCT_0915-1830.pred.h5")

data = load_h5_datasets(file_path_vec,keys=("prediction"),
                        slices={
                                # "vec": (slice(None), slice(400,600), slice(0,1000), slice(0,1000)),
                               
                                }
                        )

# vec = data["vec"]
# vol = data["volume"]


pred = data["prediction"]


# file_path_vol = Path("../data/original/subvolumes/285_01_HR_.sub-001.vol.h5")

# data = load_h5_datasets(file_path_vol,keys=("volume",),
#                         slices={"volume": ( slice(0,10), slice(0,512), slice(0,512)),}
#                         )

# vol = data["volume"]

In [ ]:

fig, ax = plt.subplots(1, 4, figsize=(16, 4))

x_slice = 5

vec[0,::] = np.abs(vec[0,::])
vec[2,::] = np.abs(vec[2,::])

# show volume
ax[0].imshow(vol[x_slice, :, :], cmap="gray")
ax[0].set_title("vol")

# common normalization for vec
vmin = np.min(vec[:, x_slice, :, :])
vmax = np.max(vec[:, x_slice, :, :])

im1 = ax[1].imshow(vec[0, x_slice, :, :], cmap="plasma", vmin=vmin, vmax=vmax)
ax[1].set_title("vec x-comp")

im2 = ax[2].imshow(vec[1, x_slice, :, :], cmap="plasma", vmin=vmin, vmax=vmax)
ax[2].set_title("vec y-comp")

im3 = ax[3].imshow(vec[2, x_slice, :, :], cmap="plasma", vmin=vmin, vmax=vmax)
ax[3].set_title("vec z-comp")

# shared colorbar for vec
fig.colorbar(im3, ax=ax[1:], shrink=0.8, location="right")

for a in ax:
    a.axis("off")


plt.show()

In [ ]:
def edge_aware_smooth_vec(v, iters=2, sigma_theta_deg=24.0, eps=1e-12):
    """
    Edge-aware smoothing for a line field (v ≡ -v). Preserves 90° jumps.
    v: (3,X,Y,Z)
    """
    v = v.copy()
    sigma_theta = np.deg2rad(sigma_theta_deg)

    def normed(x):
        n = np.linalg.norm(x, axis=0, keepdims=True)
        return x / (np.maximum(n, eps))

    v = normed(v)

    # 6-neighborhood shifts
    shifts = [(+1,0,0),(-1,0,0),(0,+1,0),(0,-1,0),(0,0,+1),(0,0,-1)]

    for _ in range(iters):
        acc = np.zeros_like(v)
        wsum = np.zeros(v.shape[1:], dtype=v.dtype)

        for dx,dy,dz in shifts:
            vn = np.roll(v, shift=(dx,dy,dz), axis=(1,2,3))

            # sign-invariant angle via abs(dot)
            c = np.abs(np.sum(v * vn, axis=0))
            c = np.clip(c, 0.0, 1.0)
            theta = np.arccos(c)

            w = np.exp(-(theta*theta) / (2*sigma_theta*sigma_theta)).astype(v.dtype)

            acc += vn * w[None, ...]
            wsum += w

        # include self weight
        acc += v
        wsum += 1.0

        v = acc / wsum[None, ...]
        v = normed(v)

    return v

In [ ]:
vec = edge_aware_smooth_vec(vec,iters=100,sigma_theta_deg=24)

In [ ]:

fig, ax = plt.subplots(1, 4, figsize=(16, 4))

x_slice = 5

# show volume
ax[0].imshow(vol[x_slice, :, :], cmap="gray")
ax[0].set_title("vol")

# common normalization for vec
vmin = np.min(vec[:, x_slice, :, :])
vmax = np.max(vec[:, x_slice, :, :])

im1 = ax[1].imshow(vec[0, x_slice, :, :], cmap="plasma", vmin=vmin, vmax=vmax)
ax[1].set_title("vec x-comp")

im2 = ax[2].imshow(vec[1, x_slice, :, :] > 0.6, cmap="plasma", vmin=vmin, vmax=vmax)
ax[2].set_title("vec y-comp")

im3 = ax[3].imshow(vec[2, x_slice, :, :] >0.9, cmap="plasma", vmin=vmin, vmax=vmax)
ax[3].set_title("vec z-comp")

# shared colorbar for vec
fig.colorbar(im1, ax=ax[1:], shrink=0.8, location="right")

for a in ax:
    a.axis("off")


plt.show()

In [ ]:
seg = np.zeros(vec.shape[1:], dtype=np.uint8)

mask1 = vec[1] > 0.5
mask2 = vec[2] > 0.5

seg[mask1 & ~mask2] = 1
seg[mask2 & ~mask1] = 2

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8,4))

ax[0].imshow(vol[x_slice], cmap="gray")
ax[0].set_title("vol")

ax[1].imshow(seg[x_slice], cmap="plasma", vmin=0, vmax=2)
ax[1].set_title("segmentation")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def _ball(radius: int) -> np.ndarray:
    if radius <= 0:
        return np.ones((1, 1, 1), dtype=bool)
    r = radius
    zz, yy, xx = np.ogrid[-r:r+1, -r:r+1, -r:r+1]
    return (zz*zz + yy*yy + xx*xx) <= r*r

def _remove_small_objects(mask: np.ndarray, min_size: int, structure=None) -> np.ndarray:
    if min_size <= 0:
        return mask
    lab, n = ndi.label(mask, structure=structure)
    if n == 0:
        return mask
    counts = np.bincount(lab.ravel())
    keep = counts >= min_size
    keep[0] = False  # background
    return keep[lab]

def _fill_small_holes(mask: np.ndarray, max_hole_size: int, structure=None) -> np.ndarray:
    """
    Fill holes (background components fully enclosed by mask) up to max_hole_size voxels.
    """
    if max_hole_size <= 0:
        return mask

    # holes are components in ~mask that do NOT touch the border
    inv = ~mask
    lab, n = ndi.label(inv, structure=structure)
    if n == 0:
        return mask

    # find labels touching the border -> not holes
    border = np.zeros_like(lab, dtype=bool)
    border[0, :, :] = True; border[-1, :, :] = True
    border[:, 0, :] = True; border[:, -1, :] = True
    border[:, :, 0] = True; border[:, :, -1] = True

    border_labels = np.unique(lab[border])
    is_hole = np.ones(n + 1, dtype=bool)
    is_hole[border_labels] = False
    is_hole[0] = False

    counts = np.bincount(lab.ravel(), minlength=n + 1)
    small_hole = is_hole & (counts <= max_hole_size)

    filled = mask.copy()
    filled[small_hole[lab]] = True
    return filled

def clean_multiclass_seg_scipy(
    seg: np.ndarray,
    classes=(1, 2),
    min_size=200,
    max_hole_size=200,
    erode_radius=1,
    class_priority=(2, 1),
    connectivity=3,  # 1=6-neigh, 2=18-neigh, 3=26-neigh in 3D
) -> np.ndarray:
    assert seg.ndim == 3, f"Expected seg (Z,Y,X), got {seg.shape}"

    # connectivity structure for labeling/morphology
    structure = ndi.generate_binary_structure(rank=3, connectivity=connectivity)
    footprint = _ball(erode_radius) if erode_radius and erode_radius > 0 else None

    cleaned_masks = {}
    for c in classes:
        m = (seg == c)

        # remove small objects
        m = _remove_small_objects(m, min_size=min_size, structure=structure)

        # fill holes up to size threshold
        m = _fill_small_holes(m, max_hole_size=max_hole_size, structure=structure)

        # erosion
        if footprint is not None:
            m = ndi.binary_erosion(m, structure=footprint)

        cleaned_masks[c] = m

    # rebuild label map; resolve overlaps by priority (first wins)
    seg_clean = np.zeros_like(seg, dtype=np.uint8)
    for c in class_priority:
        seg_clean[cleaned_masks[c]] = c

    return seg_clean

In [ ]:
# usage
seg_clean = clean_multiclass_seg_scipy(
    seg,
    min_size=200,
    max_hole_size=200,
    erode_radius=1,
    class_priority=(2, 1),
    connectivity=3,
)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8,4))

ax[0].imshow(vol[x_slice], cmap="gray")
ax[0].set_title("vol")

ax[1].imshow(seg[x_slice], cmap="plasma", vmin=0, vmax=2)
ax[1].set_title("segmentation")

ax[2].imshow(seg_clean[x_slice], cmap="plasma", vmin=0, vmax=2)
ax[2].set_title("segmentation")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()